# 实验五：语义通信 vs 传统传输（Semantic Communication）
## FMI Course · Kaggle Hands-on Lab

**课程**：未来媒体互联网（Future Media & Internet）&nbsp;|&nbsp; **预计时长**：~10 分钟（含约 2 分钟训练） &nbsp;|&nbsp; **运行环境**：Kaggle Notebook &nbsp;|&nbsp; **GPU 可选**

---

> ⚠️ **运行前请开启 Internet**：本实验使用 `torchvision.datasets.MNIST(download=True)` 自动下载 MNIST 数据集（约 11 MB）。在 Kaggle 上，请前往 **Notebook Settings → Internet → On**。数据集只需下载一次；后续再次运行时可以关闭 Internet。

## 实验概述

语义通信（Semantic Communication）是 6G 研究中的一个重要方向：它不再执着于「一比特不差地传输原始数据」，而是尝试直接传输数据的**语义信息（含义）**，从而在极端恶劣的信道条件下（极低信噪比）仍能保留信息的核心含义，即使像素/比特层面已经面目全非。

本实验在 MNIST 手写数字数据集上训练一个小型自编码器（Autoencoder），将其作为语义通信的简化模型，并与「直接对像素加噪声」的传统传输方式进行对比。你将看到：当信噪比（SNR）降到极低水平时，传统传输方式会退化为纯噪声、数字完全不可辨认，而语义通信方式却仍能保留数字的大致形状。

本实验对应课程中「语义通信与面向任务的通信」部分的核心内容，是理解 6G 愿景中「语义-任务导向通信」的一个直观入门案例。

## 学习目标

完成本实验后，你应该能够：

1. 解释语义通信与传统比特级通信在设计理念上的根本区别；
2. 理解自编码器（Autoencoder）如何被用作语义通信的简化模型（编码器提取语义、信道引入噪声、解码器恢复语义）；
3. 说明信噪比（SNR, Signal-to-Noise Ratio）的含义，并解释不同 SNR 数值（20dB / 0dB / -10dB）分别代表什么样的信道质量；
4. 从对比实验中观察并描述传统传输的「断崖式退化（Cliff Effect）」现象，以及语义通信为何能避免这一现象；
5. 说明语义通信在 6G、极低信噪比或带宽受限场景中的潜在应用价值。

## 背景与基本原理

### 传统通信的本质

传统数字通信系统的流程可以概括为：**原始数据 → 编码为比特 → 调制 → 信道传输 → 解调 → 还原比特 → 还原数据**。这套流程的设计目标是让**每一个比特**都尽可能准确地传输，整体传输质量完全由逐比特的正确率决定——只要有足够多的比特出错，恢复出的数据就会彻底损坏，与原始数据毫无关联。

### 语义通信的本质

语义通信的流程是：**原始数据 → 语义编码器（提取含义）→ 信道传输（含噪声）→ 语义解码器（恢复含义）→ 还原数据**。核心区别在于：语义编码器传输的不是原始比特，而是一个经过学习提炼的、**低维的语义表示（Semantic Representation）**。当信道引入噪声时，这个语义表示会受到干扰，但由于语义解码器是端到端训练出来的、具备一定的抗噪能力，即使表示中混入了噪声，解码器仍然可能恢复出与原始含义足够接近的结果——代价是牺牲了逐像素的精确重建。

### 自编码器作为语义通信的简化模型

本实验用一个全连接自编码器模拟语义通信系统：

- **编码器（Encoder）**：把 784 维（28×28 像素）的图像逐层压缩到 256 → 128 → 16 维的语义向量（Latent Vector），压缩比约 49:1；
- **信道（Channel）**：在语义向量上叠加与设定 SNR 相符的高斯噪声，模拟真实无线信道的噪声干扰；
- **解码器（Decoder）**：从含噪声的语义向量逐层恢复到 128 → 256 → 784 维，重建图像。

由于编码器和解码器是**联合训练**的（训练时就在语义向量上注入了噪声），解码器学会了「即使语义向量被扰动，也要尽力恢复出正确的数字形状」，这正是语义通信「鲁棒性来自于对噪声环境的联合优化」这一设计理念的体现。

### 信噪比（SNR）的含义

信噪比（Signal-to-Noise Ratio, SNR）衡量信号功率相对于噪声功率的强弱，单位为 dB：

| SNR | 含义 |
|-----|------|
| 20 dB | 信道非常干净，信号功率是噪声的 100 倍 |
| 0 dB | 信号功率与噪声功率相当 |
| -10 dB | 噪声功率是信号功率的 10 倍，信道条件极端恶劣 |

### 传统方案的「断崖效应」（Cliff Effect）

传统通信方案通常在 SNR 高于某个门限时表现良好，一旦 SNR 跌破该门限，纠错编码彻底失效，传输质量会**断崖式**下降到完全不可用——这与语义通信「优雅退化（Graceful Degradation）」的特性形成鲜明对比。

## 实验设计

**数据集**：MNIST 手写数字（0~9），是深度学习领域最经典的入门基准数据集，图像尺寸 28×28 灰度图。

**模型结构**：`SemanticAutoencoder`，编码器 784→256→128→16，信道注入噪声，解码器 16→128→256→784。

**训练配置**：在固定 SNR=10dB 的信道条件下训练 5 个 epoch（约 2 分钟 CPU 时间），让模型学会在含噪声条件下重建数字。

**对比测试**：训练完成后，在多个 SNR 水平（20 / 10 / 0 / -5 / -10 dB）下，分别测试「传统传输」（直接对原始像素加噪声）与「语义通信」（编码器→信道噪声→解码器）的重建效果。

**预期观察**：在高 SNR 下两种方式差异不大；随着 SNR 降低，传统传输会更早、更剧烈地退化，而语义通信在极低 SNR 下仍能保留数字的大致轮廓。

## 运行环境说明

| 项目 | 说明 |
|------|------|
| 运行环境 | Kaggle Notebook |
| 计算资源 | CPU（GPU 可选，可加速训练） |
| 网络访问 | 需要（首次运行需下载 MNIST，约 11 MB） |
| 主要依赖 | PyTorch、torchvision、NumPy、Matplotlib（Kaggle 已预装） |

直接点击「Run All」即可运行全部实验。首次运行前请确认已按上方提示开启 Internet。

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print('Imports OK')

## 步骤一：加载 MNIST 数据集

MNIST 是深度学习领域最标准的入门基准数据集之一，包含 6 万张 28×28 的手写数字图像。代码中使用 `download=True` 自动下载数据集到本地缓存目录 `./mnist_data`，因此**本次运行需要开启 Internet**（如上方提醒所述）。

运行后会展示 8 张随机样本及对应标签，帮助你直观熟悉数据集内容——这也是后续用来测试「语义通信 vs 传统传输」的原始数据来源。

In [ ]:
# Load MNIST data
transform = transforms.ToTensor()
train_set = torchvision.datasets.MNIST(
    root='./mnist_data', train=True, download=True, transform=transform
)
train_loader = torch.utils.data.DataLoader(train_set, batch_size=128, shuffle=True)

# Show samples
samples, labels = next(iter(train_loader))
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for i in range(8):
    axes[i].imshow(samples[i][0], cmap='gray')
    axes[i].set_title(str(labels[i].item()))
    axes[i].axis('off')
plt.suptitle('MNIST Samples')
plt.tight_layout()
plt.show()
print(f'Training set: {len(train_set)} images')

## 步骤二：定义语义通信自编码器模型

`SemanticAutoencoder` 类定义了本实验的核心模型结构：

- `encoder`：三层全连接网络，将 784 维图像压缩到 16 维的语义向量（`latent_dim=16`），压缩比约 49:1；
- `add_channel_noise()`：根据设定的 SNR（单位 dB）计算对应的噪声功率，并在语义向量上叠加符合该 SNR 的高斯噪声，模拟无线信道；
- `decoder`：三层全连接网络，将含噪声的语义向量恢复为 784 维图像；
- `forward()`：串联编码器 → 信道噪声 → 解码器，形成完整的「语义通信」前向流程。

> 提示：`latent_dim=16` 意味着原始 784 个像素值被压缩为仅 16 个数值，这 16 个数值就是模型学到的「数字的语义表示」。

In [ ]:
LATENT_DIM = 16

class SemanticAutoencoder(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(784, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, 784), nn.Sigmoid()
        )

    def add_channel_noise(self, z, snr_db):
        if snr_db is None:
            return z
        signal_power = z.pow(2).mean(dim=1, keepdim=True)
        snr_linear = 10 ** (snr_db / 10.0)
        noise_power = signal_power / snr_linear
        noise = torch.randn_like(z) * torch.sqrt(noise_power)
        return z + noise

    def forward(self, x, snr_db=10):
        z = self.encoder(x.view(x.size(0), -1))
        z_noisy = self.add_channel_noise(z, snr_db)
        return self.decoder(z_noisy).view(-1, 1, 28, 28)

model = SemanticAutoencoder().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

## 步骤三：训练模型

训练过程在 SNR=10dB 的信道条件下进行 5 个 epoch，CPU 上大约需要 2 分钟。训练目标是最小化重建图像与原始图像之间的均方误差（MSELoss）。

**关键点**：由于训练时信道噪声就已经参与了前向传播，模型学到的不是「无噪声情况下的最优压缩」，而是「在噪声存在的条件下，如何编码语义信息使得解码器依然能够正确重建」——这正是语义通信抗噪能力的来源。训练过程中打印的 Loss 应该逐 epoch 下降。

In [ ]:
# Train (about 2 minutes on CPU)
EPOCHS = 5
model.train()
for epoch in range(EPOCHS):
    total_loss = 0.0
    for batch_idx, (data, _) in enumerate(train_loader):
        data = data.to(device)
        optimizer.zero_grad()
        output = model(data, snr_db=10)
        loss = criterion(output, data)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    print(f'Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f}')
print('Training done')

## 步骤四：在不同信噪比下对比传统传输与语义通信

这部分代码选取一张测试数字图像，在 5 个 SNR 水平（20 / 10 / 0 / -5 / -10 dB）下分别运行两种传输方式：

- **传统传输（`traditional_transmit`）**：直接在原始像素上叠加符合目标 SNR 的高斯噪声；
- **语义通信**：调用训练好的 `model(img, snr_db=snr)`，即编码器 → 信道噪声 → 解码器的完整流程。

可视化结果分两行：上排为传统传输在各 SNR 下的重建图，下排为语义通信在各 SNR 下的重建图，可以直接逐列对比同一 SNR 水平下两种方式的差异。

> 思考：为什么语义通信在低 SNR 下的表现明显优于传统传输？（提示：噪声是加在哪一层——像素层还是语义层？）

In [ ]:
# Select one image for clear comparison
model.eval()
demo_loader = torch.utils.data.DataLoader(train_set, batch_size=10, shuffle=True)
test_images, test_labels = next(iter(demo_loader))
test_images = test_images.to(device)

# Pick a clear digit to demonstrate
idx = 0
img = test_images[idx:idx+1]
label = test_labels[idx].item()
print(f'Selected digit: {label}')

# Traditional: add noise directly to pixels
def traditional_transmit(img, snr_db):
    if snr_db >= 20:
        return img.clone()
    sig_pow = img.pow(2).mean()
    snr_lin = 10 ** (snr_db / 10.0)
    if snr_lin == 0:
        return img.clone()
    noise_pow = sig_pow / snr_lin
    noise = torch.randn_like(img) * torch.sqrt(noise_pow)
    return torch.clamp(img + noise, 0, 1)

# Compare at different SNR levels
snr_levels = [20, 10, 0, -5, -10]

fig, axes = plt.subplots(2, len(snr_levels) + 1, figsize=(16, 7))

# Original image
axes[0, 0].imshow(img.cpu().squeeze(), cmap='gray')
axes[0, 0].set_title(f'Original\nDigit: {label}', fontsize=10, fontweight='bold')
axes[0, 0].axis('off')
axes[1, 0].axis('off')

for j, snr in enumerate(snr_levels):
    # Traditional
    trad = traditional_transmit(img, snr)
    axes[0, j+1].imshow(trad.cpu().squeeze(), cmap='gray')
    axes[0, j+1].set_title(f'Traditional\nSNR={snr}dB', fontsize=9)
    axes[0, j+1].axis('off')

    # Semantic
    with torch.no_grad():
        sem = model(img, snr_db=snr)
    axes[1, j+1].imshow(sem.cpu().squeeze(), cmap='gray')
    axes[1, j+1].set_title(f'Semantic\nSNR={snr}dB', fontsize=9)
    axes[1, j+1].axis('off')

plt.suptitle(f'Semantic vs Traditional: Digit {label} at Different SNR\n(Top: Traditional | Bottom: Semantic)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('Key observation:')
print(f'At SNR=-10dB, traditional is pure noise (digit {label} unrecognizable)')
print(f'But semantic communication still preserves the shape of {label}')
print('Semantic comm trades pixel accuracy for meaning preservation.')

## 实验结果与分析

运行实验后，你通常会观察到以下规律：

- **SNR ≥ 0 dB**：传统传输和语义通信都能较清晰地重建数字，两者差异不大——信道条件较好时，逐比特/逐像素传输本身就足够可靠；
- **SNR = -5 dB**：传统传输已出现明显的噪点干扰，数字轮廓开始变得模糊难辨；语义通信的重建结果依然相对清晰；
- **SNR = -10 dB**：传统传输的重建图基本变成纯噪声，人眼已无法辨认出原始数字；而语义通信的重建结果虽然细节模糊、笔画不够锐利，但依然能大致辨认出数字的整体形状——这正是语义通信「保留含义、牺牲精度」设计理念的直接体现。

### 现象解读

这一现象背后的原因在于：传统传输的噪声直接叠加在**像素空间**，每个像素点都独立承受相同强度的干扰，一旦噪声超过信号强度，像素信息就彻底湮没；而语义通信的噪声叠加在**16 维的语义空间**，解码器是针对这种噪声环境专门训练出来的，它学会了利用数字类别之间的结构相关性（例如「7」和「1」在语义空间中的表示模式不同）来抵抗噪声干扰，从含噪声的语义向量中「猜出」最可能的数字形状。

### 实验局限性

需要注意：本实验的语义通信模型仅在单一 SNR（10dB）条件下训练，而测试时应用到了多个不同 SNR 水平——这是一种简化处理。真实语义通信系统通常需要在多种信噪比条件下联合训练（或使用信道自适应机制），以获得更稳健的跨 SNR 泛化能力。此外，本实验仅针对 MNIST 这类简单灰度数字图像，真实场景中的语义通信（如自然图像、语音、视频）需要更复杂的语义编码器设计。

## 从实验到实际系统

本实验演示了语义通信的核心思想，但真实的语义通信研究涉及更广泛的场景：

- **6G 中的语义通信研究**：被视为 6G 网络的候选关键技术之一，目标是在极端信道条件、超大规模连接或带宽极度受限的场景下，仍能完成高质量的信息传递；
- **超可靠低时延通信（URLLC）**：语义通信有望在保证任务完成质量的前提下，进一步压缩所需传输的数据量，从而降低时延；
- **图像/语音/视频的语义传输**：相比本实验的手写数字，真实应用场景需要针对不同模态设计专门的语义编码器（如面向人脸识别任务的语义编码器只需传输与身份相关的特征，而非完整图像）；
- **面向任务的通信（Task-Oriented Communication）**：语义通信的一个重要分支，编码器只提取「完成下游任务（如分类、检测）所必需」的信息，进一步压缩数据量，是语义通信思想的延伸。

---

## 本实验小结

通过本实验，你应该掌握以下核心结论：

1. **语义通信传输含义而非原始比特**：编码器提取语义表示，解码器从中恢复内容，而非逐比特还原；
2. **自编码器是语义通信的一个简化但有效的类比模型**：编码器/信道噪声/解码器的结构对应了「语义提取-信道传输-语义恢复」的过程；
3. **联合训练是抗噪能力的关键**：模型在训练时就暴露于信道噪声，因此学会了抵抗噪声的语义表示方式；
4. **语义通信呈现「优雅退化」而非「断崖式失效」**：在极低 SNR 下依然能保留信息的核心含义；
5. **该技术是 6G 研究的重要方向**：在极端信道条件、超低时延、面向任务的通信场景中具有独特优势。

---

## 思考与拓展

以下问题没有唯一答案，鼓励你修改代码并重新运行：

1. **调整语义向量维度**：把 `LATENT_DIM` 从 16 改为 4 或 64，观察压缩比变化对低 SNR 下重建质量的影响。
2. **延长训练时间**：把 `EPOCHS` 从 5 增加到 15 或 20，观察模型是否变得对噪声更加鲁棒。
3. **更换数据集**：尝试使用 FashionMNIST（同样通过 torchvision 加载）替代 MNIST，观察语义保留能力对非数字形状（服装轮廓）是否依然成立。
4. **多 SNR 联合训练**：修改训练循环，让每个 batch 随机采样不同的 `snr_db` 而非固定 10dB，观察模型在更大 SNR 范围内的泛化能力是否提升。
5. **量化对比**：计算传统传输与语义通信在各 SNR 下的 PSNR/SSIM（复用实验三的实现），用数值方式验证「语义通信优雅退化」的结论。

---

← [实验四：神经压缩 vs JPEG](https://www.kaggle.com/code/guopingtan/fmi-demo4-neural-compression) &nbsp;|&nbsp; 🏠 [课程主页 · Course Home](https://www.kaggle.com/code/guopingtan/fmi-course-kaggle-hands-on-lab-start-here) &nbsp;|&nbsp; [实验六：网络损伤对高清视频的影响 →](https://www.kaggle.com/code/guopingtan/fmi-demo-6-network-impairments-on-hd-video)

**FMI Course · Kaggle Hands-on Lab** &nbsp;|&nbsp; MV-AI Lab · Hohai University